<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/14-trees-and-ensembles/03_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 14: Solutions

Try the exercises first. The notes after each solution explain the idea behind it and the mistake each check is designed to catch.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import math
import warnings

import numpy as np
import pandas as pd
from scipy.stats import entropy as scipy_entropy
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import partial_dependence
from sklearn.linear_model import LinearRegression
from sklearn.metrics import log_loss, mean_squared_error
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, (pd.DataFrame, pd.Series)) or isinstance(b, (pd.DataFrame, pd.Series)):
        if type(a) is not type(b):
            return False
        options = dict(check_dtype=False, check_exact=False, rtol=tol, atol=tol, check_index_type=False)
        try:
            if isinstance(a, pd.DataFrame):
                pd.testing.assert_frame_equal(a, b, check_column_type=False, **options)
            else:
                pd.testing.assert_series_equal(a, b, **options)
            return True
        except AssertionError:
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, (pd.DataFrame, pd.Series)):
        lines = (x.to_string(max_rows=6, max_cols=6) if isinstance(x, pd.DataFrame) else x.to_string(max_rows=6)).splitlines()
        return f"{type(x).__name__} shape={x.shape}\n          " + "\n          ".join(lines[:8])
    if isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, tuple) and any(isinstance(i, np.ndarray) for i in x):
        text = "(" + ", ".join(_short(i, limit) for i in x) + ")"
        return text
    else:
        text = repr(x)
    if isinstance(x, (pd.DataFrame, pd.Series)):
        return text
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def _gini_ref(labels):
    _, counts = np.unique(np.asarray(labels), return_counts=True)
    p = counts / counts.sum()
    return float(1 - np.sum(p ** 2))


def _weighted_gini_ref(left, right):
    left, right = np.asarray(left), np.asarray(right)
    return (len(left) * _gini_ref(left) + len(right) * _gini_ref(right)) / (len(left) + len(right))


def _predict_ref(tree, X):
    out = []
    for row in np.asarray(X, dtype=float):
        node = tree
        while "value" not in node:
            node = node["left"] if row[node["feature"]] <= node["threshold"] else node["right"]
        out.append(node["value"])
    return np.array(out)


def check_gini_impurity(f):
    rng = np.random.default_rng(0)
    labels = rng.choice(["a", "b", "c"], 500, p=[0.5, 0.3, 0.2])
    p = pd.Series(labels).value_counts(normalize=True).to_numpy()
    _scenarios("gini_impurity", [
        ("a pure group is 0", lambda: f(["a", "a", "a"]), 0.0, None),
        ("two equal classes give 0.5", lambda: f([0, 1, 0, 1]), 0.5, "1 - (0.5² + 0.5²)"),
        ("three equal classes give 2/3", lambda: f(["x", "y", "z"]), 2 / 3, None),
        ("a 90/10 group", lambda: f([1] * 9 + [0]), 1 - 0.81 - 0.01, None),
        ("random labels", lambda: f(labels), float(1 - np.sum(p ** 2)), None),
        ("bools and arrays work", lambda: f(np.array([True, False, True, True])), 1 - 0.75 ** 2 - 0.25 ** 2, None),
        ("returns a float", lambda: type(f([1, 2])).__name__, "float", None),
        ("an empty group raises", lambda: f([]), Raises(ValueError), None),
    ])


def check_entropy(f):
    _scenarios("entropy", [
        ("a pure group is exactly 0.0", lambda: (f(["a"] * 5), math.copysign(1, f(["a"] * 5))), (0.0, 1.0),
         "-0.0 is not allowed: write it as Σ p log₂(1/p)"),
        ("two equal classes give 1 bit", lambda: f([1, 1, 0, 0]), 1.0, None),
        ("four equal classes give 2 bits", lambda: f(["a", "b", "c", "d"]), 2.0, "use log base 2"),
        ("a 90/10 group", lambda: f([1] * 9 + [0]), -(0.9 * math.log2(0.9) + 0.1 * math.log2(0.1)), None),
        ("matches scipy", lambda: f([0, 0, 1, 2, 2, 2]), float(scipy_entropy([2, 1, 3], base=2)), None),
        ("returns a float", lambda: type(f([1, 2])).__name__, "float", None),
        ("an empty group raises", lambda: f([]), Raises(ValueError), None),
    ])


def check_weighted_impurity(f):
    rng = np.random.default_rng(1)
    y = rng.integers(0, 3, 200)

    def never_worse_than_parent():
        parent = 1 - np.sum((np.bincount(y) / len(y)) ** 2)
        for _ in range(50):
            mask = rng.random(200) < rng.random()
            if f(y[mask], y[~mask]) > parent + 1e-12:
                return False
        return True

    _scenarios("weighted_impurity", [
        ("a perfect split gives 0", lambda: f([0, 0, 0], [1, 1]), 0.0, None),
        ("weights by size", lambda: f([0, 1], [0, 0, 0, 0, 0, 0, 0, 1]), (2 * 0.5 + 8 * (1 - 0.875 ** 2 - 0.125 ** 2)) / 10,
         "(n_left · I(left) + n_right · I(right)) / n"),
        ("entropy criterion", lambda: f([0, 1], [1, 1, 1, 1], criterion="entropy"), 2 / 6 * 1.0, None),
        ("an empty side contributes nothing", lambda: f([], [0, 1]), 0.5, None),
        ("never worse than the parent", never_worse_than_parent, True, None),
        ("returns a float", lambda: type(f([0], [1])).__name__, "float", None),
        ("an unknown criterion raises", lambda: f([0], [1], criterion="mse"), Raises(ValueError), None),
        ("two empty sides raise", lambda: f([], []), Raises(ValueError), None),
    ])


def check_predict_with_tree(f):
    example = {"feature": 0, "threshold": 2.5,
               "left": {"value": "small"},
               "right": {"feature": 1, "threshold": 10, "left": {"value": "medium"}, "right": {"value": "large"}}}
    rng = np.random.default_rng(2)
    X = rng.normal(size=(200, 3))
    y = (X[:, 0] + X[:, 1] * X[:, 2] > 0).astype(int)
    sk = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X, y)

    def to_dict(node=0):
        t = sk.tree_
        if t.children_left[node] == -1:
            return {"value": int(sk.classes_[np.argmax(t.value[node][0])])}
        return {"feature": int(t.feature[node]), "threshold": float(t.threshold[node]),
                "left": to_dict(t.children_left[node]), "right": to_dict(t.children_right[node])}

    X_new = rng.normal(size=(300, 3))
    _scenarios("predict_with_tree", [
        ("the example", lambda: f(example, [[1, 50], [3, 5], [3, 20]]).tolist(), ["small", "medium", "large"],
         "go left when x[feature] <= threshold"),
        ("a value equal to the threshold goes left", lambda: f(example, [[2.5, 99]]).tolist(), ["small"], "use <=, not <"),
        ("a single leaf", lambda: f({"value": 7}, [[1, 2], [3, 4]]).tolist(), [7, 7], None),
        ("a DataFrame works", lambda: f(example, pd.DataFrame({"a": [0, 4], "b": [0, 0]})).tolist(), ["small", "medium"], None),
        ("returns an array", lambda: type(f(example, [[0, 0]])).__name__, "ndarray", None),
        ("matches a scikit-learn tree", lambda: f(to_dict(), X_new).tolist(), sk.predict(X_new).tolist(), None),
    ])


def check_best_split(f):
    rng = np.random.default_rng(3)
    x = np.round(rng.normal(size=400), 3)
    y = np.where(x + rng.normal(0, 0.7, 400) > 0.4, "yes", "no")
    sk = DecisionTreeClassifier(max_depth=1, random_state=0).fit(x[:, None], y)
    x_pos = np.exp(x)
    _scenarios("best_split", [
        ("a small example", lambda: f([1, 2, 3, 4], [0, 0, 1, 1]), (2.5, 0.0), "thresholds are midpoints between distinct values"),
        ("ties go to the smallest threshold", lambda: f([1, 2, 3], [0, 1, 0]), (1.5, 1 / 3), None),
        ("repeated x values", lambda: f([1, 1, 2, 2, 3], ["a", "a", "b", "a", "b"]),
         (1.5, _weighted_gini_ref(["a", "a"], ["b", "a", "b"])), None),
        ("matches scikit-learn's root split", lambda: (round(f(x, y)[0], 6), round(f(x, y)[1], 6)),
         (round(float(sk.tree_.threshold[0]), 6), round(_weighted_gini_ref(y[x <= sk.tree_.threshold[0]], y[x > sk.tree_.threshold[0]]), 6)), None),
        ("only the order of x matters", lambda: np.sum(x_pos <= f(x_pos, y)[0]), int(np.sum(x <= sk.tree_.threshold[0])), None),
        ("one distinct value: no split", lambda: f([5, 5, 5], [0, 1, 1]), (None, 1 - (1 / 3) ** 2 - (2 / 3) ** 2), None),
    ])


def check_best_split_regression(f):
    rng = np.random.default_rng(4)
    x = rng.uniform(0, 10, 20_000)
    y = np.where(x < 6.2, 1.0, 3.0) + np.sin(x) + rng.normal(0, 0.5, 20_000)
    sk = DecisionTreeRegressor(max_depth=1, random_state=0).fit(x[:, None], y)
    x_small = np.round(rng.uniform(0, 1, 300), 2)
    y_small = 2 * x_small + rng.normal(0, 0.2, 300)

    def brute(xv, yv):
        best = (None, np.inf)
        values = np.unique(xv)
        for t in (values[:-1] + values[1:]) / 2:
            left, right = yv[xv <= t], yv[xv > t]
            mse = (np.sum((left - left.mean()) ** 2) + np.sum((right - right.mean()) ** 2)) / len(yv)
            if mse < best[1] - 1e-12:
                best = (float(t), float(mse))
        return best

    _scenarios("best_split_regression", [
        ("a small example", lambda: f([1, 2, 3, 4], [5.0, 5.0, 9.0, 9.0]), (2.5, 0.0), None),
        ("repeated x values", lambda: f([1, 1, 2, 3], [0.0, 2.0, 10.0, 10.0]), (1.5, 0.5), "never split between equal x values"),
        ("ties go to the smallest threshold", lambda: f([1, 2, 3], [0.0, 5.0, 0.0]), (1.5, 12.5 / 3), None),
        ("matches brute force with many repeated values", lambda: f(x_small, y_small), brute(x_small, y_small), None),
        ("20,000 points: matches scikit-learn's threshold", lambda: round(f(x, y)[0], 4), round(float(sk.tree_.threshold[0]), 4),
         "sort once, then use cumulative sums"),
        ("one distinct value: no split", lambda: f([2, 2, 2], [1.0, 2.0, 3.0]), (None, 2 / 3), None),
    ])


def check_bagging_with_oob(f):
    rng = np.random.default_rng(5)
    X = rng.uniform(-2, 2, size=(300, 3))
    y = X[:, 0] ** 2 + np.sin(2 * X[:, 1]) + rng.normal(0, 0.3, 300)
    X_new = rng.uniform(-2, 2, size=(2000, 3))
    y_new = X_new[:, 0] ** 2 + np.sin(2 * X_new[:, 1]) + rng.normal(0, 0.3, 2000)

    def reference(n_trees, seed=0, Xn=X_new):
        g = np.random.default_rng(seed)
        new_total, oob_total, oob_count = np.zeros(len(Xn)), np.zeros(len(y)), np.zeros(len(y))
        for _ in range(n_trees):
            idx = g.integers(0, len(y), len(y))
            tree = DecisionTreeRegressor(random_state=0).fit(X[idx], y[idx])
            new_total += tree.predict(Xn)
            oob = np.ones(len(y), dtype=bool)
            oob[idx] = False
            oob_total[oob] += tree.predict(X[oob])
            oob_count[oob] += 1
        with np.errstate(invalid="ignore", divide="ignore"):
            return new_total / n_trees, np.where(oob_count > 0, oob_total / oob_count, np.nan)

    ref = reference(50)
    result = {}

    def run():
        if "v" not in result:
            result["v"] = f(X, y, X_new, 50)
        return result["v"]

    def oob_close_to_test():
        new, oob = run()
        oob_rmse = np.sqrt(np.nanmean((y - oob) ** 2))
        test_rmse = np.sqrt(np.mean((y_new - new) ** 2))
        return bool(abs(oob_rmse - test_rmse) / test_rmse < 0.15)

    _scenarios("bagging_with_oob", [
        ("predictions for new data match the reference", lambda: run()[0], ref[0],
         "idx = rng.integers(0, n, n) for each tree, from one rng"),
        ("out-of-bag predictions match the reference", lambda: np.nan_to_num(run()[1], nan=-999.0), np.nan_to_num(ref[1], nan=-999.0), None),
        ("the seed is used", lambda: f(X, y, X_new[:5], 10, seed=3)[0], reference(10, seed=3, Xn=X_new[:5])[0], None),
        ("with 2 trees, rows seen by both are nan", lambda: int(np.isnan(f(X, y, X_new[:5], 2)[1]).sum()),
         int(np.isnan(reference(2, Xn=X_new[:5])[1]).sum()), "use np.nan when a row was never out of bag"),
        ("the OOB error is close to the test error", oob_close_to_test, True, None),
    ])


def check_permutation_importance_scores(f):
    rng = np.random.default_rng(6)
    X = pd.DataFrame({"strong": rng.normal(size=600), "weak": rng.normal(size=600), "noise": rng.normal(size=600)})
    y = 3 * X["strong"] + 0.7 * X["weak"] + rng.normal(0, 0.5, 600)
    model = LinearRegression().fit(X[:400], y[:400])
    X_test, y_test = X[400:].reset_index(drop=True), y[400:].reset_index(drop=True)

    def reference(n_repeats=5, seed=0):
        baseline = mean_squared_error(y_test, model.predict(X_test))
        g = np.random.default_rng(seed)
        out = {}
        for column in X_test.columns:
            values = []
            for _ in range(n_repeats):
                Xs = X_test.copy()
                Xs[column] = g.permutation(Xs[column].to_numpy())
                values.append(mean_squared_error(y_test, model.predict(Xs)) - baseline)
            out[column] = float(np.mean(values))
        return pd.Series(out)

    before = X_test.copy()

    def unchanged():
        f(model, X_test, y_test, mean_squared_error)
        return bool(X_test.equals(before))

    _scenarios("permutation_importance_scores", [
        ("matches the reference procedure", lambda: f(model, X_test, y_test, mean_squared_error), reference(),
         "one rng; columns in order; n_repeats shuffles each"),
        ("the seed and repeats are used", lambda: f(model, X_test, y_test, mean_squared_error, n_repeats=3, seed=9), reference(3, 9), None),
        ("ranks strong > weak > noise", lambda: list(f(model, X_test, y_test, mean_squared_error).sort_values(ascending=False).index),
         ["strong", "weak", "noise"], None),
        ("X is not changed", unchanged, True, "shuffle a copy"),
    ])


def check_partial_dependence_1d(f):
    rng = np.random.default_rng(7)
    X = pd.DataFrame({"a": rng.uniform(0, 10, 400), "b": rng.normal(size=400), "c": rng.integers(0, 5, 400).astype(float)})
    y = np.sin(X["a"]) + X["b"] * (X["c"] > 2) + rng.normal(0, 0.2, 400)
    forest = RandomForestRegressor(n_estimators=30, random_state=0).fit(X, y)
    linear = LinearRegression().fit(X, y)
    ref = partial_dependence(forest, X, ["a"], method="brute", grid_resolution=15)
    grid = ref["grid_values"][0]
    before = X.copy()

    def unchanged():
        f(forest, X, "b", [0.0, 1.0])
        return bool(X.equals(before))

    _scenarios("partial_dependence_1d", [
        ("matches scikit-learn for a forest", lambda: f(forest, X, "a", grid), ref["average"][0], None),
        ("a linear model gives a straight line with the weight as slope",
         lambda: np.diff(f(linear, X, "b", [0.0, 1.0, 2.0])), np.full(2, linear.coef_[1]), None),
        ("returns floats", lambda: f(forest, X, "c", [1, 2]).dtype.kind, "f", None),
        ("X is not changed", unchanged, True, "work on a copy"),
    ])


def check_build_tree(f):
    rng = np.random.default_rng(8)
    X = rng.normal(size=(250, 4))
    y = ((X[:, 0] + X[:, 1] ** 2 + rng.normal(0, 0.5, 250)) > 1).astype(int)
    X3 = rng.normal(size=(240, 3))
    y3 = np.select([X3[:, 0] > 0.5, X3[:, 1] > 0], ["a", "b"], "c")
    y3 = np.where(rng.random(240) < 0.1, "c", y3)

    def compare(Xd, yd, depth, leaf):
        sk = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf, random_state=0).fit(Xd, yd)
        tree = f(Xd, yd, depth, leaf)
        return bool((_predict_ref(tree, Xd) == sk.predict(Xd)).all())

    def depth_of(node):
        return 0 if "value" in node else 1 + max(depth_of(node["left"]), depth_of(node["right"]))

    def smallest_leaf(node, rows):
        if "value" in node:
            return len(rows)
        go_left = X[rows, node["feature"]] <= node["threshold"]
        return min(smallest_leaf(node["left"], rows[go_left]), smallest_leaf(node["right"], rows[~go_left]))

    _scenarios("build_tree", [
        ("a stump on a small example", lambda: f([[1], [2], [3], [4]], [0, 0, 1, 1], max_depth=1),
         {"feature": 0, "threshold": 2.5, "left": {"value": 0}, "right": {"value": 1}}, None),
        ("a pure node is a leaf", lambda: f([[1], [2]], [1, 1], max_depth=3), {"value": 1}, None),
        ("leaf values are plain Python values", lambda: type(f([[1], [2]], [1, 1], max_depth=3)["value"]).__name__, "int",
         "use .item() on the NumPy value"),
        ("ties in a leaf go to the smallest label", lambda: f([[1], [1]], ["b", "a"], max_depth=2), {"value": "a"}, None),
        ("depth 2 matches scikit-learn", lambda: compare(X, y, 2, 1), True, None),
        ("depth 4 matches scikit-learn", lambda: compare(X, y, 4, 1), True, None),
        ("min_samples_leaf=10 matches scikit-learn", lambda: compare(X, y, 5, 10), True, "both sides need at least min_samples_leaf rows"),
        ("three string classes match scikit-learn", lambda: compare(X3, y3, 3, 5), True, None),
        ("max_depth is respected", lambda: depth_of(f(X, y, 3)) <= 3, True, None),
        ("min_samples_leaf is respected", lambda: smallest_leaf(f(X, y, 6, 15), np.arange(len(y))) >= 15, True, None),
    ])


def check_gradient_boosting_regressor(f):
    rng = np.random.default_rng(9)
    X = rng.uniform(-2, 2, size=(400, 3))
    y = X[:, 0] ** 2 + np.sin(2 * X[:, 1]) * X[:, 2] + rng.normal(0, 0.2, 400)
    X_new = rng.uniform(-2, 2, size=(1000, 3))
    y_new = X_new[:, 0] ** 2 + np.sin(2 * X_new[:, 1]) * X_new[:, 2] + rng.normal(0, 0.2, 1000)

    def reference(n, rate, depth, Xn):
        start = y.mean()
        current = np.full(len(y), start)
        out = np.full(len(Xn), start)
        for _ in range(n):
            tree = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(X, y - current)
            current += rate * tree.predict(X)
            out += rate * tree.predict(Xn)
        return out

    def training_error_falls():
        errors = [np.mean((y - f(X, y, n, 0.1, 2)(X)) ** 2) for n in [1, 5, 20, 60]]
        return bool(all(a > b for a, b in zip(errors, errors[1:])))

    def close_to_sklearn():
        sk = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0).fit(X, y)
        ours = np.sqrt(np.mean((y_new - f(X, y, 100, 0.1, 3)(X_new)) ** 2))
        theirs = np.sqrt(np.mean((y_new - sk.predict(X_new)) ** 2))
        return bool(abs(ours - theirs) / theirs < 0.1)

    _scenarios("gradient_boosting_regressor", [
        ("zero trees predict the mean", lambda: f(X, y, 0, 0.1, 2)(X_new[:3]), np.full(3, y.mean()), "start from y.mean()"),
        ("matches the reference", lambda: f(X, y, 50, 0.1, 3)(X_new), reference(50, 0.1, 3, X_new),
         "fit each tree to y - current predictions, and add learning_rate × its predictions"),
        ("another setting", lambda: f(X, y, 20, 0.5, 1)(X_new), reference(20, 0.5, 1, X_new), None),
        ("the training error falls with more trees", training_error_falls, True, None),
        ("learning rate 1 with unlimited depth memorises at once", lambda: float(np.max(np.abs(y - f(X, y, 1, 1.0, None)(X)))) < 1e-8,
         True, None),
        ("close to scikit-learn's GradientBoostingRegressor", close_to_sklearn, True, None),
    ])


def check_binary_gradient_boosting(f):
    rng = np.random.default_rng(10)
    X = rng.normal(size=(500, 3))
    y = (rng.random(500) < 1 / (1 + np.exp(-(1.5 * X[:, 0] - X[:, 1] ** 2 + 0.5)))).astype(int)
    X_new = rng.normal(size=(800, 3))

    def reference(n, rate, depth, Xn):
        rate_1 = y.mean()
        start = np.log(rate_1 / (1 - rate_1))
        F, F_new = np.full(len(y), start), np.full(len(Xn), start)
        for _ in range(n):
            p = 1 / (1 + np.exp(-F))
            r = y - p
            tree = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(X, r)
            leaves = tree.apply(X)
            values = {leaf: r[leaves == leaf].sum() / (p[leaves == leaf] * (1 - p[leaves == leaf])).sum() for leaf in np.unique(leaves)}
            F += rate * np.array([values[leaf] for leaf in leaves])
            F_new += rate * np.array([values[leaf] for leaf in tree.apply(Xn)])
        return 1 / (1 + np.exp(-F_new))

    def loss_falls():
        losses = [log_loss(y, f(X, y, n, 0.1, 2)(X)) for n in [1, 5, 20, 60]]
        return bool(all(a > b for a, b in zip(losses, losses[1:])))

    def close_to_sklearn():
        sk = GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=0).fit(X, y)
        return bool(np.max(np.abs(f(X, y, 50, 0.1, 3)(X_new) - sk.predict_proba(X_new)[:, 1])) < 0.1)

    _scenarios("binary_gradient_boosting", [
        ("zero trees predict the base rate", lambda: f(X, y, 0, 0.1, 2)(X_new[:3]), np.full(3, y.mean()), "start from the log-odds of the base rate"),
        ("matches the reference", lambda: f(X, y, 30, 0.1, 3)(X_new), reference(30, 0.1, 3, X_new),
         "leaf value = Σ r / Σ p(1 - p) over the training examples in the leaf"),
        ("another setting", lambda: f(X, y, 10, 0.5, 1)(X_new), reference(10, 0.5, 1, X_new), None),
        ("probabilities are between 0 and 1", lambda: bool(((lambda p: (p > 0) & (p < 1))(f(X, y, 20, 0.3, 3)(X_new))).all()), True, None),
        ("the training log loss falls", loss_falls, True, None),
        ("close to scikit-learn's GradientBoostingClassifier", close_to_sklearn, True, None),
        ("one class only raises", lambda: f(X, np.ones(500), 5, 0.1, 2), Raises(ValueError), None),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `gini_impurity`

In [2]:
def gini_impurity(labels) -> float:
    labels = np.asarray(labels)
    if labels.size == 0:
        raise ValueError("an empty group has no impurity")
    _, counts = np.unique(labels, return_counts=True)
    p = counts / counts.sum()
    return float(1 - np.sum(p ** 2))

In [3]:
check_gini_impurity(gini_impurity)

PASS  all 8 tests passed for gini_impurity


`np.unique(..., return_counts=True)` gives each class once with how often it appears, so the fractions are one division away. It works for any label type, which matters because tree code shouldn't care whether classes are `0/1` or `"Adelie"/"Gentoo"`.

The Gini impurity has a nice meaning: pick two labels at random from the group (with replacement); `G` is the chance they're **different**. For a pure group that's 0. For `k` equally common classes it's `1 - 1/k`: 0.5 for two classes, about 0.667 for three. The check tests both.

Note that `Σ p_k²` is the chance two random labels are the **same**, so `G = 1 - P(same)`. That's the "concentration" economists call the Herfindahl index, which shows up in measuring how concentrated a market is.

### Exercise 2: `entropy`

In [4]:
def entropy(labels) -> float:
    labels = np.asarray(labels)
    if labels.size == 0:
        raise ValueError("an empty group has no entropy")
    _, counts = np.unique(labels, return_counts=True)
    p = counts / counts.sum()
    return float(np.sum(p * np.log2(1 / p)))

In [5]:
check_entropy(entropy)

PASS  all 7 tests passed for entropy


Writing it as `Σ p log₂(1/p)` instead of `-Σ p log₂ p` gives the same number, but each term is already positive, so a pure group gives `1 · log₂(1) = 0.0` rather than `-0.0`. (Python prints `-0.0` faithfully, and it looks like a bug in a report.)

Because `np.unique` only returns classes that appear, every `p` is positive and `log₂` never sees a zero. If you counted classes some other way, for example with a fixed list of all possible classes, you'd need to skip zero counts yourself, since `0 · log 0` is defined as 0 but NumPy computes it as `nan`.

Entropy is the average number of yes/no questions needed to identify a label, if you ask cleverly (Module 9). Four equally common classes need 2 questions, which is why the example gives 2.0 bits. A decision tree using `criterion="entropy"` picks the split that reduces this the most, which is called the **information gain**.

### Exercise 3: `weighted_impurity`

In [6]:
def weighted_impurity(y_left, y_right, criterion: str = "gini") -> float:
    measures = {"gini": gini_impurity, "entropy": entropy}
    if criterion not in measures:
        raise ValueError(f"criterion must be 'gini' or 'entropy', got {criterion!r}")
    measure = measures[criterion]
    y_left, y_right = np.asarray(y_left), np.asarray(y_right)
    n_left, n_right = len(y_left), len(y_right)
    if n_left + n_right == 0:
        raise ValueError("both groups are empty")
    total = 0.0
    if n_left:
        total += n_left * measure(y_left)
    if n_right:
        total += n_right * measure(y_right)
    return float(total / (n_left + n_right))

In [7]:
check_weighted_impurity(weighted_impurity)

PASS  all 8 tests passed for weighted_impurity


A dictionary from name to function picks the measure without an `if`/`elif` chain, and makes adding a third criterion a one-line change (functions are objects you can store and pass around, Module 1).

Why weight by size? A split that isolates one unusual example in a perfectly pure group of 1 hasn't achieved much if the other 999 examples are as mixed as before. Weighting by size means the big group's impurity dominates, which is what you want.

The **improvement** of a split is the parent's impurity minus this weighted impurity. It's never negative: splitting a group can't make the weighted impurity higher than the group's own (the check tests this on random splits). That's a property of concave functions like Gini and entropy, and it's why a tree always has *some* split that doesn't hurt, even when none really helps. A tree has to decide when an improvement is too small to be worth it, which is what the stopping settings in the lesson are for.

### Exercise 4: `predict_with_tree`

In [8]:
def predict_with_tree(tree: dict, X) -> np.ndarray:
    X = np.asarray(X, dtype=float)

    def predict_one(row):
        node = tree
        while "value" not in node:
            node = node["left"] if row[node["feature"]] <= node["threshold"] else node["right"]
        return node["value"]

    return np.array([predict_one(row) for row in X])

In [9]:
check_predict_with_tree(predict_with_tree)

PASS  all 6 tests passed for predict_with_tree


Each prediction walks from the root down to a leaf, one comparison per level, so predicting is fast: a tree of depth 20 answers in at most 20 comparisons, however much data it was trained on.

A `while` loop is used instead of recursion. Recursion works too (`return predict_one(row, node["left"])`), and is arguably closer to the shape of the data, but very deep trees could hit Python's recursion limit of about 1,000 levels.

`np.asarray(X, dtype=float)` makes DataFrames and lists behave the same, using column **positions**. That's how scikit-learn stores trees internally as well: `tree_.feature` holds positions, not names. It's one reason why the columns of new data must be in the same order as the training data.

The `<=` matters: scikit-learn sends values **equal** to the threshold left, and the check tests a value exactly on a threshold.

## Medium

### Exercise 5: `best_split`

In [10]:
def best_split(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y)
    values = np.unique(x)                      # sorted distinct values
    if len(values) < 2:
        return None, gini_impurity(y)
    best_threshold, best_score = None, np.inf
    for threshold in (values[:-1] + values[1:]) / 2:
        go_left = x <= threshold
        score = weighted_impurity(y[go_left], y[~go_left])
        if score < best_score:                 # strict: keeps the smallest threshold on ties
            best_threshold, best_score = float(threshold), score
    return best_threshold, float(best_score)

In [11]:
check_best_split(best_split)

PASS  all 6 tests passed for best_split


Why midpoints? Any threshold between two neighbouring values splits the training data the same way, so the choice only matters for **new** data that falls in the gap. The midpoint is the neutral choice, and it's what scikit-learn uses.

Using `<` rather than `<=` when comparing scores means a later threshold only wins if it's strictly better, so on ties the earliest (smallest) threshold is kept.

This version is simple but slow for large data: it recomputes the impurity from scratch for every candidate, so `n` distinct values cost about `n²` work. Real implementations sort once and sweep from left to right, updating the class counts on each side as each example crosses over, which costs `n log n`. Exercise 6 does exactly that for regression. Histogram-based boosting goes further and only considers 255 candidate thresholds per feature (lesson section 10).

Notice that the result only depends on the **order** of the `x` values: taking the logarithm of `x` would move the threshold but split the data identically (the check tests it). That's why trees don't need scaling or log transforms.

### Exercise 6: `best_split_regression`

In [12]:
def best_split_regression(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(y)
    order = np.argsort(x, kind="stable")
    xs, ys = x[order], y[order]
    # positions i where a split can go: between xs[i - 1] and xs[i], with the values different
    cut = np.flatnonzero(xs[1:] != xs[:-1]) + 1
    if len(cut) == 0:
        return None, float(np.var(y))

    sum_y, sum_y2 = np.cumsum(ys), np.cumsum(ys ** 2)
    n_left = cut
    left_sum, left_sum2 = sum_y[cut - 1], sum_y2[cut - 1]
    right_sum, right_sum2 = sum_y[-1] - left_sum, sum_y2[-1] - left_sum2
    sse_left = left_sum2 - left_sum ** 2 / n_left
    sse_right = right_sum2 - right_sum ** 2 / (n - n_left)
    mse = (sse_left + sse_right) / n

    best = int(np.argmin(mse))                 # first minimum: the smallest threshold
    return float((xs[cut[best] - 1] + xs[cut[best]]) / 2), float(mse[best])

In [13]:
check_best_split_regression(best_split_regression)

PASS  all 6 tests passed for best_split_regression


The identity `SSE = Σy² - (Σy)²/n` is what makes this fast: with running totals of `y` and `y²`, the SSE of the left group for every possible cut is available at once, and the right group's totals are the overall totals minus the left's. One sort plus a few vectorised operations, instead of recomputing a variance for every candidate.

This is exactly how real tree libraries search for regression splits, and why their cost grows like `n log n` rather than `n²`.

A word of caution about the identity: it subtracts two large, nearly equal numbers when the values of `y` are large and close together (for example house prices around 500,000 that differ by a few dollars), which can lose precision (Module 9's floating-point section). Libraries guard against this, for example by subtracting the mean of `y` first. It's fine for the data here.

Minimising the weighted MSE is the same as maximising the reduction in variance, which is how this criterion is often described.

### Exercise 7: `bagging_with_oob`

In [14]:
from sklearn.tree import DecisionTreeRegressor


def bagging_with_oob(X, y, X_new, n_trees: int, seed: int = 0):
    X, y, X_new = np.asarray(X, dtype=float), np.asarray(y, dtype=float), np.asarray(X_new, dtype=float)
    n = len(y)
    rng = np.random.default_rng(seed)
    new_total = np.zeros(len(X_new))
    oob_total = np.zeros(n)
    oob_count = np.zeros(n)
    for _ in range(n_trees):
        idx = rng.integers(0, n, n)
        tree = DecisionTreeRegressor(random_state=0).fit(X[idx], y[idx])
        new_total += tree.predict(X_new)
        out_of_bag = np.ones(n, dtype=bool)
        out_of_bag[idx] = False
        if out_of_bag.any():
            oob_total[out_of_bag] += tree.predict(X[out_of_bag])
            oob_count[out_of_bag] += 1
    with np.errstate(invalid="ignore", divide="ignore"):
        oob = np.where(oob_count > 0, oob_total / oob_count, np.nan)
    return new_total / n_trees, oob

In [15]:
check_bagging_with_oob(bagging_with_oob)

PASS  all 5 tests passed for bagging_with_oob


The out-of-bag mask starts as all `True`, and every row that was drawn (even several times) is set to `False`. Only the remaining rows are predicted by this tree.

Each training row is out of bag for about 37% of the trees, so with 100 trees it gets an average of about 37 predictions from trees that never saw it. That's an honest prediction, just like a cross-validated one, and you get it as a side effect of training. The check confirms that the OOB error is close to the error on fresh test data, while the error of the same forest on its own training data is far too optimistic.

With very few trees, some rows are drawn by every tree and have no OOB prediction at all; `np.nan` marks them honestly instead of pretending. scikit-learn warns about this situation (`oob_score=True` with too few trees).

`RandomForestRegressor(max_features=1.0, oob_score=True)` does all of this in fast compiled code, in parallel, and keeps the result in `oob_prediction_`. It uses a different random number scheme, so its numbers won't match yours exactly.

### Exercise 8: `permutation_importance_scores`

In [16]:
def permutation_importance_scores(model, X, y, metric, n_repeats: int = 5, seed: int = 0) -> pd.Series:
    baseline = metric(y, model.predict(X))
    rng = np.random.default_rng(seed)
    importances = {}
    for column in X.columns:
        increases = []
        for _ in range(n_repeats):
            X_shuffled = X.copy()
            X_shuffled[column] = rng.permutation(X_shuffled[column].to_numpy())
            increases.append(metric(y, model.predict(X_shuffled)) - baseline)
        importances[column] = float(np.mean(increases))
    return pd.Series(importances)

In [17]:
check_permutation_importance_scores(permutation_importance_scores)

PASS  all 4 tests passed for permutation_importance_scores


Shuffling a column keeps its distribution exactly (same values, same spread) but breaks its connection with the target and with the other columns. If the model's error goes up, the model was using that column.

`X.copy()` matters: shuffling the caller's DataFrame in place would silently corrupt their data for everything that comes after. The check tests that `X` is unchanged.

Some points worth knowing:

- Use **held-out** data. On training data, a model that memorised a noise column will look like it depends on it, which is the same bias as impurity importance (lesson section 8).
- Repeat the shuffle a few times: one shuffle is noisy, especially for small data.
- A column can have a slightly **negative** importance. That means shuffling it happened to help a little, which is noise; read it as "not important".
- With correlated columns, shuffling one leaves the model able to use the other, so both look less important than they are.

scikit-learn's `sklearn.inspection.permutation_importance` does the same with any scorer, in parallel, and reports the spread over repeats too.

### Exercise 9: `partial_dependence_1d`

In [18]:
def partial_dependence_1d(model, X, feature: str, grid) -> np.ndarray:
    averages = []
    X_changed = X.copy()
    for value in grid:
        X_changed[feature] = value
        averages.append(model.predict(X_changed).mean())
    return np.array(averages, dtype=float)

In [19]:
check_partial_dependence_1d(partial_dependence_1d)

PASS  all 4 tests passed for partial_dependence_1d


A partial dependence curve answers "what does the model predict, on average over the data, if this feature had value `v`?". It shows the **shape** of a feature's effect: rising, falling, flattening off, jumping at a threshold. For a linear model it's a straight line with the feature's weight as its slope; for a tree ensemble it's a step-like curve that can bend however the data suggests.

One copy is enough: each loop overwrites the whole column, so earlier values don't leak into later ones.

Two cautions:

- **Averages hide differences.** If a feature raises predictions for some examples and lowers them for others, the average can be flat. Plotting the individual curves (one per example, called **ICE** curves, `kind="individual"` in scikit-learn's `PartialDependenceDisplay`) shows this.
- **Impossible combinations.** Setting a feature to a value for every row creates rows that may never occur in reality, such as a 10-room house with 1 bedroom per room. When features are strongly correlated, the curve partly reflects how the model behaves on such unrealistic data.

The check uses `method="brute"` because for some tree models scikit-learn has a faster method (`"recursion"`) that uses the tree structure directly and gives slightly different values.

## Hard

### Exercise 10: `build_tree`

In [20]:
def build_tree(X, y, max_depth: int, min_samples_leaf: int = 1) -> dict:
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)

    def leaf(labels):
        classes, counts = np.unique(labels, return_counts=True)
        return {"value": classes[np.argmax(counts)].item()}     # argmax picks the first, smallest label on ties

    def grow(rows, depth):
        labels = y[rows]
        node_gini = gini_impurity(labels)
        if depth == max_depth or node_gini == 0 or len(rows) < 2 * min_samples_leaf:
            return leaf(labels)

        best_score, best_feature, best_threshold = node_gini, None, None
        for j in range(X.shape[1]):
            order = rows[np.argsort(X[rows, j], kind="stable")]
            xs, ys = X[order, j], y[order]
            for i in range(min_samples_leaf, len(order) - min_samples_leaf + 1):
                if xs[i - 1] == xs[i]:
                    continue                    # no threshold separates equal values
                score = (i * gini_impurity(ys[:i]) + (len(ys) - i) * gini_impurity(ys[i:])) / len(ys)
                if score < best_score:
                    best_score, best_feature, best_threshold = score, j, (xs[i - 1] + xs[i]) / 2
        if best_feature is None:
            return leaf(labels)

        go_left = X[rows, best_feature] <= best_threshold
        return {"feature": int(best_feature), "threshold": float(best_threshold),
                "left": grow(rows[go_left], depth + 1), "right": grow(rows[~go_left], depth + 1)}

    return grow(np.arange(len(y)), 0)

In [21]:
check_build_tree(build_tree)

PASS  all 10 tests passed for build_tree


The tree is built by a **recursive** function: `grow` decides whether the current group of rows becomes a leaf; if not, it finds the best split and calls itself on each half. The recursion stops at the leaves. Trees are the classic example of data where recursion fits naturally.

Passing **row positions** (`rows`) instead of copies of the data keeps memory use low: each level just selects positions from the original arrays.

The split search sorts the rows by each feature once, then tries each cut position `i`, skipping positions between equal values. Requiring `i` to run from `min_samples_leaf` to `len - min_samples_leaf` enforces the leaf size directly. Starting `best_score` at the node's own Gini means only strictly improving splits are accepted.

This is the same greedy algorithm as scikit-learn's (called CART, from Breiman, Friedman, Olshen and Stone's 1984 book), which is why the predictions match. scikit-learn's version is much faster: it's written in Cython, updates class counts incrementally instead of recomputing the Gini for every cut, and handles details like sample weights and missing values. Ours costs roughly `n²` per feature per node, which is fine for a few hundred rows and far too slow for a million.

When features have **tied** values or two splits are equally good, scikit-learn can pick a different but equally good split, because it visits features in a random order. That's why the check uses continuous random features.

### Exercise 11: `gradient_boosting_regressor`

In [22]:
from sklearn.tree import DecisionTreeRegressor


def gradient_boosting_regressor(X, y, n_estimators: int, learning_rate: float, max_depth: int):
    X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
    start = y.mean()
    current = np.full(len(y), start)
    trees = []
    for _ in range(n_estimators):
        tree = DecisionTreeRegressor(max_depth=max_depth, random_state=0).fit(X, y - current)
        current += learning_rate * tree.predict(X)
        trees.append(tree)

    def predict(X_new):
        X_new = np.asarray(X_new, dtype=float)
        prediction = np.full(len(X_new), start)
        for tree in trees:
            prediction += learning_rate * tree.predict(X_new)
        return prediction

    return predict

In [23]:
check_gradient_boosting_regressor(gradient_boosting_regressor)

PASS  all 6 tests passed for gradient_boosting_regressor


Returning a function is a neat way to package a trained model: `predict` is a **closure** (Module 4) that remembers `start`, `trees` and `learning_rate`. In a larger program you'd write a class with `fit` and `predict` methods (Module 3), which is what scikit-learn does, but the idea is the same.

Each tree is fitted to the **current residuals**, the mistakes of everything so far. Shallow trees (depth 2 to 5) are typical: each one only needs to capture a small correction, and the depth limits how many features can interact within one tree. A depth-1 tree (a stump) can't model interactions at all, so the whole model becomes a sum of one-feature effects.

The check confirms some properties beyond matching the reference:

- The **training** error goes down with every round, since each tree reduces the squared error of the current residuals (or at worst leaves it unchanged).
- With `learning_rate=1` and `max_depth=None`, the first tree memorises the residuals completely, and the training error drops to (almost) zero after one round: boosting without a brake overfits immediately.
- Your results are close to scikit-learn's `GradientBoostingRegressor`, which follows the same algorithm with a slightly different split criterion.

### Exercise 12: `binary_gradient_boosting`

In [24]:
from sklearn.tree import DecisionTreeRegressor


def binary_gradient_boosting(X, y, n_estimators: int, learning_rate: float, max_depth: int):
    X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
    if y.min() == y.max():
        raise ValueError("y must contain both classes")
    rate = y.mean()
    start = np.log(rate / (1 - rate))
    F = np.full(len(y), start)
    stages = []
    for _ in range(n_estimators):
        p = 1 / (1 + np.exp(-F))
        residuals = y - p
        tree = DecisionTreeRegressor(max_depth=max_depth, random_state=0).fit(X, residuals)
        leaves = tree.apply(X)
        leaf_values = {}
        for leaf in np.unique(leaves):
            in_leaf = leaves == leaf
            leaf_values[leaf] = residuals[in_leaf].sum() / (p[in_leaf] * (1 - p[in_leaf])).sum()
        F += learning_rate * np.array([leaf_values[leaf] for leaf in leaves])
        stages.append((tree, leaf_values))

    def predict_proba(X_new):
        X_new = np.asarray(X_new, dtype=float)
        F_new = np.full(len(X_new), start)
        for tree, leaf_values in stages:
            F_new += learning_rate * np.array([leaf_values[leaf] for leaf in tree.apply(X_new)])
        return 1 / (1 + np.exp(-F_new))

    return predict_proba

In [25]:
check_binary_gradient_boosting(binary_gradient_boosting)

PASS  all 7 tests passed for binary_gradient_boosting


This is how gradient boosting handles classification, and it shows the "gradient" in the name doing real work:

- The model adds up trees on the **log-odds** scale, where any real number is allowed, and the sigmoid turns the total into a probability. That's the same link as logistic regression (Module 12); boosting just replaces the weighted sum of features with a sum of trees.
- For the log loss, the negative gradient with respect to `F` is `y - p`, so each tree is fitted to the difference between the labels and the current probabilities.
- The tree's own leaf values (the average residual) would be a plain gradient step. The **Newton step** divides the summed gradient by the summed second derivative, `p(1 - p)`, which accounts for curvature, exactly like Newton's method for Platt scaling in Module 13. Where predictions are already confident (`p` near 0 or 1), the curvature is small and the same residual produces a bigger step.

`tree.apply(X)` returns the **leaf index** each example lands in, which is what makes it possible to keep the tree's structure but replace its values.

This is essentially what scikit-learn's `GradientBoostingClassifier` does, and XGBoost and LightGBM go one step further by also choosing the **splits** using the second derivatives. The check confirms that the training log loss goes down round by round and that your probabilities are close to scikit-learn's.

A real implementation guards against a leaf where every `p(1 - p)` is tiny (dividing by almost zero gives a huge step). With the data in the check, that doesn't happen.

## What's next

The mini project in `04_mini_project.ipynb` returns to the California housing data from Module 12: a single tree, a random forest and gradient boosting against the best linear model, compared fairly on familiar and unseen regions, and then explained with permutation importance and partial dependence.